# 04_tools

对应 `index.md` 第 4 阶段：`@tool`、工具 schema、绑定 / 调用工具。

笔记本在 `codes/langchain/04_tools/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT` 并加载 `codes/langchain/local.env`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

本阶段起不再使用 FakeModel。真实模型：`init_chat_model("deepseek:...")`，密钥 `DEEPSEEK_API_KEY`（可选 `DEEPSEEK_MODEL`）。


In [1]:
import os
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "04_tools" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "langchain" / "04_tools"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

def load_local_env(env_path: Path) -> None:
    """把 local.env 写入 os.environ（已有键不覆盖）。"""
    if not env_path.is_file():
        return
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip("\"'")
        if key and key not in os.environ:
            os.environ[key] = value

ROOT = lab_root()
load_local_env(ROOT.parent / "local.env")
ROOT, bool(os.environ.get("DEEPSEEK_API_KEY"))


(PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/langchain/04_tools'),
 True)

## 1. 用 @tool 定义并调用

用 `@tool` 定义函数 `mul(a: int, b: int) -> int`，文档字符串写 `两数相乘。`。

用字典 `{"a": 3, "b": 4}` 调用它，打印返回值。


In [6]:
from langchain_core.tools import tool

# 作答

@tool
def mul(a: int, b: int) -> int:
    """两数相乘。"""
    return a * b

@tool(description="两数相加")
def add(a: int, b: int) -> int:
    return a + b

print(mul.invoke({"a": 3, "b": 4}))
print(add.invoke({"a": 3, "b": 4}))
#评阅
# 对。mul 有 docstring，invoke 得 12。多写的 add 无妨。

#参考答案
# @tool
# def mul(a: int, b: int) -> int:
#     """两数相乘。"""
#     return a * b
# print(mul.invoke({"a": 3, "b": 4}))

12
7


## 2. 看工具的名字、说明、参数

已有工具 `add`。依次打印：`add.name`、`add.description`、以及 `list(add.args.keys())`。


In [9]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b

# 作答


print(add.name)
print(add.description)
print(list(add.args.keys()))
from rich import print as rprint
rprint(dir(add))
#评阅
# 对。name / description / args.keys 都打印对了。

#参考答案
# print(add.name)
# print(add.description)
# print(list(add.args.keys()))

add
两数相加。
['a', 'b']


[
    'InputType',
    'OutputType',
    '__abstractmethods__',
    '__annotate_func__',
    '__annotations__',
    '__annotations_cache__',
    '__class__',
    '__class_getitem__',
    '__class_vars__',
    '__copy__',
    '__deepcopy__',
    '__delattr__',
    '__dict__',
    '__dir__',
    '__doc__',
    '__eq__',
    '__fields__',
    '__fields_set__',
    '__firstlineno__',
    '__format__',
    '__ge__',
    '__get_pydantic_core_schema__',
    '__get_pydantic_json_schema__',
    '__getattr__',
    '__getattribute__',
    '__getstate__',
    '__gt__',
    '__hash__',
    '__init__',
    '__init_subclass__',
    '__iter__',
    '__le__',
    '__lt__',
    '__module__',
    '__ne__',
    '__new__',
    '__or__',
    '__orig_bases__',
    '__parameters__',
    '__pretty__',
    '__private_attributes__',
    '__pydantic_complete__',
    '__pydantic_computed_fields__',
    '__pydantic_core_schema__',
    '__pydantic_custom_init__',
    '__pydantic_decorators__',
    '__pydantic_extra__',
    '__pydantic_extra_info__',
    '__pydantic_fields__',
    '__pydantic_fields_set__',
    '__pydantic_generic_metadata__',
    '__pydantic_init_subclass__',
    '__pydantic_on_complete__',
    '__pydantic_parent_namespace__',
    '__pydantic_post_init__',
    '__pydantic_private__',
    '__pydantic_root_model__',
    '__pydantic_serializer__',
    '__pydantic_setattr_handlers__',
    '__pydantic_validator__',
    '__reduce__',
    '__reduce_ex__',
    '__replace__',
    '__repr__',
    '__repr_args__',
    '__repr_name__',
    '__repr_recursion__',
    '__repr_str__',
    '__rich_repr__',
    '__ror__',
    '__setattr__',
    '__setstate__',
    '__signature__',
    '__sizeof__',
    '__slots__',
    '__static_attributes__',
    '__str__',
    '__subclasshook__',
    '__weakref__',
    '_abatch_with_config',
    '_abc_impl',
    '_acall_with_config',
    '_arun',
    '_astream_events_v1_v2',
    '_astream_events_v3_unsupported',
    '_atransform_stream_with_config',
    '_batch_with_config',
    '_calculate_keys',
    '_call_with_config',
    '_copy_and_set_values',
    '_filter_injected_args',
    '_get_value',
    '_injected_args_keys',
    '_iter',
    '_parse_input',
    '_run',
    '_setattr_handler',
    '_to_args_and_kwargs',
    '_transform_stream_with_config',
    'abatch',
    'abatch_as_completed',
    'ainvoke',
    'args',
    'args_schema',
    'arun',
    'as_tool',
    'assign',
    'astream',
    'astream_events',
    'astream_log',
    'atransform',
    'batch',
    'batch_as_completed',
    'bind',
    'callbacks',
    'config_schema',
    'config_specs',
    'configurable_alternatives',
    'configurable_fields',
    'construct',
    'copy',
    'coroutine',
    'description',
    'dict',
    'extras',
    'from_function',
    'from_orm',
    'func',
    'get_config_jsonschema',
    'get_graph',
    'get_input_jsonschema',
    'get_input_schema',
    'get_lc_namespace',
    'get_name',
    'get_output_jsonschema',
    'get_output_schema',
    'get_prompts',
    'handle_tool_error',
    'handle_validation_error',
    'input_schema',
    'invoke',
    'is_lc_serializable',
    'is_single_input',
    'json',
    'lc_attributes',
    'lc_id',
    'lc_secrets',
    'map',
    'metadata',
    'model_computed_fields',
    'model_config',
    'model_construct',
    'model_copy',
    'model_dump',
    'model_dump_json',
    'model_extra',
    'model_fields',
    'model_fields_set',
    'model_json_schema',
    'model_parametrized_name',
    'model_post_init',
    'model_rebuild',
    'model_validate',
    'model_validate_json',
    'model_validate_strings',
    'name',
    'output_schema',
    'parse_file',
    'parse_obj',
    'parse_raw',
    'pick',
    'pipe',
    'response_format',
    'return_direct',
    'run',
    'schema',
    'schema_json',
    'stream',
    'stream_events',
    'tags',
    'to_json',
    'to_json_not_implemented',
    'tool_call_schema',
    'transform',
    'update_forward_refs',
    'validate',
    

## 3. 从 tool_calls 跑工具并回 ToolMessage

已有工具 `add`，以及带 `tool_calls` 的 `ai`。

取出第一条 tool_call：用它的 `args` 调用 `add`，再用返回值构造 `ToolMessage`（`tool_call_id` 用该 call 的 `id`，`content` 转成字符串）。

打印 `ToolMessage` 的 `content` 和 `tool_call_id`。


In [11]:
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b

ai = AIMessage(
    content="",
    tool_calls=[
        {"name": "add", "args": {"a": 2, "b": 5}, "id": "call_1", "type": "tool_call"}
    ],
)

# 作答

result = add.invoke(ai.tool_calls[0]["args"])
tool_message = ToolMessage(
    content=f"Result is {result}",
    tool_call_id=ai.tool_calls[0]["id"],    
)
print(tool_message.content)
print(tool_message.tool_call_id)
#评阅
# tool_call_id 对。content 题意是返回值转字符串（如 "7"），不必加 "Result is" 前缀。

#参考答案
# call = ai.tool_calls[0]
# out = add.invoke(call["args"])
# msg = ToolMessage(content=str(out), tool_call_id=call["id"])
# print(msg.content, msg.tool_call_id)


Result is 7
call_1


## 4. 按名字分发工具

已有工具列表 `tools` 和一条 `tool_call`（字典，含 `name` / `args`）。

先做成 `name -> tool` 的字典，再按 `tool_call["name"]` 找到工具并 `invoke(tool_call["args"])`，打印结果。


In [12]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b

@tool
def mul(a: int, b: int) -> int:
    """两数相乘。"""
    return a * b

tools = [add, mul]
tool_call = {"name": "mul", "args": {"a": 3, "b": 5}, "id": "call_2"}

# 作答

tool_map = {
    "add": add,
    "mul": mul,
}

result = tool_map[tool_call["name"]].invoke(tool_call["args"])
print(result)
#评阅
# 结果 15 对。更好从 tools 推导：`{t.name: t for t in tools}`，不要手写死键名。

#参考答案
# tools_by_name = {t.name: t for t in tools}
# print(tools_by_name[tool_call["name"]].invoke(tool_call["args"]))

15


## 5. bind_tools + 真模型发起 tool_call

已有工具 `add`。用 `init_chat_model` 创建聊天模型（默认 `deepseek:deepseek-v4-flash`，或读 `DEEPSEEK_MODEL` 拼成 `deepseek:{name}`）。

对模型 `bind_tools([add])`，再 `invoke` 一条用户消息，要求模型**调用工具**计算 `2+3`。

打印返回消息的 `tool_calls`（应至少有一条，名字为 `add`）。


In [21]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b

# 作答

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash"
)

model = model.bind_tools([add])

try:
    result = model.invoke([HumanMessage(content="必须使用给定的add工具，计算 2+3")])
except Exception as e:
    print(e)

print(result.tool_calls)
#评阅
# 对。bind_tools 后模型给出 add(2,3) 的 tool_calls。

#参考答案
# name = os.environ.get("DEEPSEEK_MODEL", "deepseek-v4-flash")
# model = init_chat_model(f"deepseek:{name}").bind_tools([add])
# ai = model.invoke([HumanMessage("用工具计算 2+3")])
# print(ai.tool_calls)

[{'name': 'add', 'args': {'a': 2, 'b': 3}, 'id': 'call_00_j45jlmoAeKxzjkGVJ3IY8418', 'type': 'tool_call'}]
